# Digital Image Processing Lab - Assignment 5
**Problem Statements:**
1. **Write a program to enhance the contrast of the input grayscale image using histogram equalization.** Display the original image and equalized image.
2. **Write a program to perform histogram matching of the input image with that of reference image using histogram equalization.**

---
### Theoretical Principles
1. **Histogram Equalization:**
   Spreads out the most frequent intensity values over the entire range $[0, L-1]$:
   - Probability Density Function (PDF): $p_r(r_k) = \frac{n_k}{M \times N}$
   - Cumulative Distribution Function (CDF): $s_k = T(r_k) = (L - 1) \sum_{j=0}^{k} p_r(r_j)$

2. **Histogram Matching (Specification):**
   Generates a processed image whose histogram matches a target reference distribution:
   - Equalize input image: $s = T(r)$
   - Equalize reference image: $v = G(z)$
   - Invert reference mapping: for each input level $r_k$, find target level $z_q$ such that $G(z_q) \approx T(r_k)$.


In [ ]:
# Cell 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
import cv2

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['image.cmap'] = 'gray'


In [ ]:
# Cell 2: Test Images Creation (Input & Reference)
def generate_images():
    # Input image: underexposed / dark image
    x = np.linspace(-2, 2, 256)
    y = np.linspace(-2, 2, 256)
    xx, yy = np.meshgrid(x, y)
    pattern = np.cos(xx) * np.sin(yy)
    dark_img = np.clip((pattern * 40 + 60), 0, 255).astype(np.uint8)
    cv2.putText(dark_img, 'INPUT', (70, 130), cv2.FONT_HERSHEY_SIMPLEX, 1.2, 120, 3)

    # Reference image: high contrast image with bimodal distribution
    ref_img = np.zeros((256, 256), dtype=np.uint8)
    cv2.circle(ref_img, (128, 128), 90, 220, -1)
    cv2.rectangle(ref_img, (40, 40), (100, 100), 180, -1)
    cv2.putText(ref_img, 'REF', (90, 140), cv2.FONT_HERSHEY_SIMPLEX, 1.2, 30, 3)
    
    return dark_img, ref_img

input_img, ref_img = generate_images()
print(f"Input shape={input_img.shape}, Ref shape={ref_img.shape}")


In [ ]:
# Cell 3: Task 1 - Histogram Equalization From Scratch
def histogram_equalization_scratch(image):
    """Performs histogram equalization from scratch using PDF and CDF."""
    h, w = image.shape
    num_pixels = h * w
    
    # Step 1: Compute histogram
    hist = np.bincount(image.ravel(), minlength=256)
    
    # Step 2: Compute PDF
    pdf = hist / num_pixels
    
    # Step 3: Compute CDF
    cdf = np.cumsum(pdf)
    
    # Step 4: Compute transformation map: s_k = round(255 * cdf)
    transform_map = np.round(255.0 * cdf).astype(np.uint8)
    
    # Step 5: Map pixels
    equalized = transform_map[image]
    return equalized, hist, cdf, transform_map

eq_scratch, orig_hist, orig_cdf, t_map = histogram_equalization_scratch(input_img)

# Verify against OpenCV cv2.equalizeHist
eq_opencv = cv2.equalizeHist(input_img)
print(f"Maximum pixel discrepancy with OpenCV: {np.max(np.abs(eq_scratch.astype(int) - eq_opencv.astype(int)))}")


In [ ]:
# Cell 4: Task 2 - Histogram Matching (Specification) From Scratch
def histogram_matching_scratch(source_img, target_img):
    """Matches the histogram of source_img to target_img."""
    # Get CDFs of both images
    _, _, src_cdf, src_map = histogram_equalization_scratch(source_img)
    _, _, tgt_cdf, tgt_map = histogram_equalization_scratch(target_img)
    
    # For each level s in src_map, find target level z where tgt_map[z] is closest to src_map[s]
    # G(z) approx T(r)
    lookup_table = np.zeros(256, dtype=np.uint8)
    
    for r in range(256):
        s_val = src_map[r]
        # Find index in tgt_map that minimizes difference
        diff = np.abs(tgt_map.astype(int) - int(s_val))
        closest_z = np.argmin(diff)
        lookup_table[r] = closest_z
        
    matched_img = lookup_table[source_img]
    return matched_img, lookup_table

matched_img, lut = histogram_matching_scratch(input_img, ref_img)
print("Histogram matching completed successfully.")


In [ ]:
# Cell 5: Visualizations
# Plot Task 1: Histogram Equalization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].imshow(input_img)
axes[0, 0].set_title("Original Image (Low Contrast)", fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(eq_scratch)
axes[0, 1].set_title("Equalized Image (Enhanced Contrast)", fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

axes[1, 0].hist(input_img.ravel(), bins=256, range=[0, 256], color='navy', alpha=0.7)
axes[1, 0].set_title("Original Histogram", fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(eq_scratch.ravel(), bins=256, range=[0, 256], color='darkorange', alpha=0.7)
axes[1, 1].set_title("Equalized Histogram (Flattened)", fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Plot Task 2: Histogram Matching
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

axes[0, 0].imshow(input_img)
axes[0, 0].set_title("Input Image", fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(ref_img)
axes[0, 1].set_title("Reference Image", fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

axes[0, 2].imshow(matched_img)
axes[0, 2].set_title("Histogram Matched Output", fontsize=12, fontweight='bold')
axes[0, 2].axis('off')

axes[1, 0].hist(input_img.ravel(), bins=256, range=[0, 256], color='navy', alpha=0.7)
axes[1, 0].set_title("Input Histogram", fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(ref_img.ravel(), bins=256, range=[0, 256], color='forestgreen', alpha=0.7)
axes[1, 1].set_title("Reference Target Histogram", fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].hist(matched_img.ravel(), bins=256, range=[0, 256], color='crimson', alpha=0.7)
axes[1, 2].set_title("Matched Output Histogram", fontweight='bold')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Conclusions
- Histogram equalization automatically balances dynamic range across the full gray scale without human intervention.
- Histogram matching provides precise control over the output histogram shape, enabling transfer of tonal qualities and lighting conditions from a reference image onto a target image.
